In [ ]:
# === CareConnect bootstrap — run me first ===
# Makes this notebook work from any folder and gives a clear error if the
# lab_helpers package is missing (e.g. not uploaded to the repo).
import os, sys

def _find_repo_root(start=None):
    here = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(here, "lab_helpers")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            return None
        here = parent

_root = _find_repo_root()
if _root is None:
    raise RuntimeError(
        "Could not find the 'lab_helpers/' folder from " + os.getcwd() + ".\n"
        "This means the helper package is not next to the notebooks.\n"
        "Fix: make sure lab_helpers/ and requirements.txt are in the same folder\n"
        "as these .ipynb files (see README > Setup). Then re-run this cell.")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

In [ ]:
# === Preflight: confirm every helper file is present BEFORE running the lab ===
import os
_required = [
    "requirements.txt",
    "lab_helpers/__init__.py",
    "lab_helpers/utils.py",
    "lab_helpers/careconnect_agents.py",
    "lab_helpers/deterministic_safety.py",
    "lab_helpers/runtime_entrypoint.py",
]
_missing = [f for f in _required if not os.path.isfile(f)]
if _missing:
    raise RuntimeError("Missing required files:\n  - " + "\n  - ".join(_missing) +
        "\n\nUpload the full lab_helpers/ folder + requirements.txt, then re-run.")
print("Preflight OK — all helper files present.")

# Lab 5: Supervisor + Deploy to AgentCore Runtime

The Supervisor orchestrates safety → retrieval → escalation → verification with step/time
budgets. We then deploy it to **AgentCore Runtime** using the `bedrock-agentcore-starter-toolkit`
`Runtime()` class — **entirely from Python**, no Node CLI.

The deployable entrypoint lives in `lab_helpers/runtime_entrypoint.py`.

### Step 1: Run the Supervisor locally first

In [ ]:
import importlib
import lab_helpers.runtime_entrypoint as rt
importlib.reload(rt)
import asyncio

async def ask(q):
    return await rt.invoke({"prompt": q})

print(asyncio.get_event_loop().run_until_complete(
    ask("What are the visiting hours at Riverside Health?")))
print("---")
print(asyncio.get_event_loop().run_until_complete(
    ask("I have a colonoscopy next Tuesday and I am almost out of my metformin. "
        "How do I prepare, can I get my refill in time, and should I stop taking it?")))

### Step 2: Deploy to AgentCore Runtime with the starter toolkit

This builds a container, pushes to ECR, and creates the managed runtime — all from Python.

In [ ]:
import boto3
from bedrock_agentcore_starter_toolkit import Runtime
import lab_helpers.utils as u

exec_role = u.create_agentcore_runtime_execution_role()
runtime = Runtime()

runtime.configure(
    entrypoint="lab_helpers/runtime_entrypoint.py",
    execution_role=exec_role,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=u.REGION,
    agent_name=u.RUNTIME_AGENT_NAME,
)
print("Configured runtime.")

In [ ]:
launch_result = runtime.launch()
print("Launched:", launch_result.agent_arn)
u.put_ssm_parameter(f"{u.SSM_PREFIX}/runtime_arn", launch_result.agent_arn)

In [ ]:
import time
while True:
    st = runtime.status()
    ep = getattr(st, "endpoint", None)
    print("status:", ep.get("status") if ep else "provisioning...")
    if ep and ep.get("status") in ("READY", "ACTIVE"):
        break
    time.sleep(20)

### Step 3: Invoke the deployed runtime

In [ ]:
import json
agentcore = boto3.client("bedrock-agentcore", region_name=u.REGION)
arn = u.get_ssm_parameter(f"{u.SSM_PREFIX}/runtime_arn")
resp = agentcore.invoke_agent_runtime(
    agentRuntimeArn=arn,
    payload=json.dumps({"prompt": "How should I prepare for my colonoscopy?"}).encode())
print(resp["response"].read().decode())

## Lab 5 complete ✅

Supervisor deployed to a managed AgentCore Runtime endpoint, invoked from the SDK.